# SunSight Stage 3 (GPU Colab/Kaggle): Auto-Labeling 360 Tile Esri dengan Microsoft Buildings + SAM 2.1

Cukup unggah **`raw_esri_bogor.zip`** (berisi folder `images/` dan `metadata.json` hasil Stage 1 — **tidak perlu pre-label di laptop!**). Notebook ini memanfaatkan koneksi cepat datacenter Colab/Kaggle + GPU T4 untuk:
1. Mengunduh **Microsoft Global ML Building Footprints** Kota Bogor (`QuadKey 310012113`) dalam ~5 detik.
2. Memproyeksikan seluruh poligon bangunan Kota Bogor ke 360 tile Esri sebagai *prompt* kotak awal.
3. Menjalankan **SAM 2.1 (`sam2.1_b.pt`)** agar setiap poligon atap menempel ketat di piksel Esri (menghilangkan geser koordinat, memisahkan rumah permukiman padat per kavling, dan menangkap spandek biru/hijau/asbes/dak beton).
4. Mengekspor `labels_sam2_refined.zip` (format YOLO-Seg) + `previews_sam2.zip` untuk QC cepat.

In [ ]:
# 1. Install dependensi (Ultralytics sudah membawa SAM 2.1 secara native)
!pip install -q ultralytics opencv-python-headless shapely requests tqdm

In [ ]:
# 2. Ekstrak raw_esri_bogor.zip (upload dulu ke /content/raw_esri_bogor.zip atau mount Google Drive)
import zipfile
from pathlib import Path

WORK_DIR = Path('/content/sunsight_autolabel')
WORK_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = Path('/content/raw_esri_bogor.zip')
if not ZIP_PATH.exists():
    raise FileNotFoundError('Upload raw_esri_bogor.zip (berisi images/ dan metadata.json) ke /content/ terlebih dahulu!')

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(WORK_DIR)

# Deteksi folder root (jika ZIP punya subfolder raw_esri_bogor/)
if (WORK_DIR / 'raw_esri_bogor' / 'metadata.json').exists():
    DATA_ROOT = WORK_DIR / 'raw_esri_bogor'
else:
    DATA_ROOT = WORK_DIR

print('Data siap di:', DATA_ROOT)
print('Jumlah gambar:', len(list((DATA_ROOT / 'images').glob('*.jpg'))))

In [ ]:
# 3. Unduh Microsoft Global ML Building Footprints Kota Bogor (QuadKey 310012113) di datacenter Colab (~5 detik)
import csv, gzip, io, json, math
import requests
from shapely.geometry import Polygon, box
from shapely.strtree import STRtree

MIN_LNG, MIN_LAT, MAX_LNG, MAX_LAT = 106.730, -6.670, 106.855, -6.520
LINKS_URL = 'https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv'

resp = requests.get(LINKS_URL, timeout=60)
resp.raise_for_status()

target_url = None
for row in csv.DictReader(io.StringIO(resp.text)):
    if (row.get('QuadKey') or '').strip() == '310012113':
        url = (row.get('Url') or '').strip()
        if 'indonesia' in (row.get('Location') or '').lower() or 'regionname=indonesia' in url.lower():
            target_url = url
            break
        if target_url is None:
            target_url = url

gz_path = WORK_DIR / 'ms_buildings_310012113.csv.gz'
if not gz_path.exists():
    print('Mengunduh:', target_url)
    with requests.get(target_url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with gz_path.open('wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

print(f'File terunduh: {gz_path.stat().st_size / 1e6:.1f} MB. Membangun indeks spasial Kota Bogor...')
bogor_rings, bogor_geoms = [], []
with gzip.open(gz_path, 'rt', encoding='utf-8', errors='ignore') as gz:
    for line in gz:
        if ('106.7' not in line and '106.8' not in line) or ('-6.5' not in line and '-6.6' not in line):
            continue
        try:
            feat = json.loads(line)
        except json.JSONDecodeError:
            continue
        geom = feat.get('geometry') or {}
        if geom.get('type') != 'Polygon':
            continue
        coords = (geom.get('coordinates') or [[]])[0]
        if len(coords) < 3:
            continue
        lng0, lat0 = float(coords[0][0]), float(coords[0][1])
        if not (MIN_LNG <= lng0 <= MAX_LNG and MIN_LAT <= lat0 <= MAX_LAT):
            continue
        poly = Polygon([(float(p[0]), float(p[1])) for p in coords])
        if not poly.is_valid:
            poly = poly.buffer(0)
        if not poly.is_empty and poly.geom_type == 'Polygon':
            bogor_geoms.append(poly)
            bogor_rings.append([(float(p[1]), float(p[0])) for p in coords])

bldg_tree = STRtree(bogor_geoms)
print(f'Siap! Total poligon bangunan Kota Bogor di indeks spasial: {len(bogor_geoms):,}')

In [ ]:
# 4. Jalankan SAM 2.1 Refinement menggunakan bbox Microsoft Buildings (+18% pad kompensasi geser Esri)
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import SAM

TILE_SIZE = 256
IMAGES_DIR = DATA_ROOT / 'images'
OUT_LABELS_DIR = WORK_DIR / 'labels_sam2_refined'
OUT_PREVIEWS_DIR = WORK_DIR / 'previews_sam2'
OUT_LABELS_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREVIEWS_DIR.mkdir(parents=True, exist_ok=True)

records = json.loads((DATA_ROOT / 'metadata.json').read_text(encoding='utf-8'))
sam_model = SAM('sam2.1_b.pt')

def latlng_to_px(lat, lng, start_x, start_y, zoom):
    n = 2 ** zoom
    rad = math.radians(lat)
    fx = (lng + 180.0) / 360.0 * n
    fy = (1.0 - math.log(math.tan(rad) + 1.0 / math.cos(rad)) / math.pi) / 2.0 * n
    return (fx - start_x) * TILE_SIZE, (fy - start_y) * TILE_SIZE

def get_prompt_bboxes_for_tile(rec, pad_ratio=0.18):
    min_lng, min_lat, max_lng, max_lat = rec['bbox']
    idxs = bldg_tree.query(box(min_lng, min_lat, max_lng, max_lat), predicate='intersects')
    w, h = rec['width'], rec['height']
    bboxes = []
    for idx in idxs:
        ring = bogor_rings[int(idx)]
        pts = [latlng_to_px(lat, lng, rec['start_x'], rec['start_y'], rec['zoom']) for lat, lng in ring]
        xs = [max(0.0, min(float(w - 1), p[0])) for p in pts]
        ys = [max(0.0, min(float(h - 1), p[1])) for p in pts]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        bw, bh = x2 - x1, y2 - y1
        if bw < 8 or bh < 8:
            continue
        px, py = bw * pad_ratio, bh * pad_ratio
        bboxes.append([
            max(0.0, x1 - px), max(0.0, y1 - py),
            min(float(w - 1), x2 + px), min(float(h - 1), y2 + py),
        ])
    return bboxes

def mask_to_clean_yolo_line(mask_u8, w, h, min_area_px=140.0):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    clean = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)
    clean = cv2.morphologyEx(clean, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area_px:
        return None, None
    (_, _), (rw, rh), _ = cv2.minAreaRect(cnt)
    if min(rw, rh) <= 0 or (max(rw, rh) / min(rw, rh)) > 4.5:
        return None, None
    eps = 0.012 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, eps, True).reshape(-1, 2)
    if len(approx) < 3:
        return None, None
    tokens = ['0']
    for x, y in approx:
        tokens.append(f'{max(0.0, min(1.0, x / w)):.6f}')
        tokens.append(f'{max(0.0, min(1.0, y / h)):.6f}')
    return ' '.join(tokens), approx.astype(np.int32)

total_refined = 0
for rec in tqdm(records, desc='SAM 2.1 Auto-Labeling 360 Tile'):
    img_path = IMAGES_DIR / rec['filename']
    if not img_path.exists():
        continue
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        continue
    h, w = bgr.shape[:2]
    bboxes = get_prompt_bboxes_for_tile(rec)
    yolo_lines = []
    preview = bgr.copy()
    overlay = bgr.copy()

    # Jalankan SAM 2.1 dalam batch kecil (maks 64 kotak per panggilan agar VRAM T4 selalu aman)
    for i in range(0, len(bboxes), 64):
        chunk_boxes = bboxes[i : i + 64]
        results = sam_model.predict(str(img_path), bboxes=chunk_boxes, verbose=False)
        if results and results[0].masks is not None:
            masks_np = results[0].masks.data.cpu().numpy()
            for m in masks_np:
                m_u8 = cv2.resize((m > 0.5).astype(np.uint8) * 255, (w, h), interpolation=cv2.INTER_NEAREST)
                line, poly = mask_to_clean_yolo_line(m_u8, w, h)
                if line is not None:
                    yolo_lines.append(line)
                    cv2.fillPoly(overlay, [poly], (0, 210, 255))
                    cv2.polylines(preview, [poly], True, (0, 230, 255), 2)

    (OUT_LABELS_DIR / f"{rec['id']}.txt").write_text('\n'.join(yolo_lines), encoding='utf-8')
    blended = cv2.addWeighted(overlay, 0.25, preview, 0.75, 0)
    cv2.imwrite(str(OUT_PREVIEWS_DIR / rec['filename']), blended, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    total_refined += len(yolo_lines)

print(f'Selesai! Total poligon atap ter-refine SAM 2.1: {total_refined:,}')

In [ ]:
# 5. Paketkan hasil label YOLO-Seg + gambar preview untuk diunduh kembali ke lokal / Roboflow
import shutil

shutil.make_archive('/content/labels_sam2_refined', 'zip', WORK_DIR, 'labels_sam2_refined')
shutil.make_archive('/content/previews_sam2', 'zip', WORK_DIR, 'previews_sam2')
print('Siap diunduh:')
print('  - /content/labels_sam2_refined.zip')
print('  - /content/previews_sam2.zip')